In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd


# Prompt the user to select a MALLET run folder (must contain 'input_filenames.txt').
# Searches recursively from the current working directory to find valid runs.
def choose_mallet_run(
    root=".",
    prompt="Select a MALLET run folder (contains 'input_filenames.txt'):"
):
    """Recursively find and select a MALLET run folder."""
    run_candidates = []

    for r, dirs, files in os.walk(root):
        if ".ipynb_checkpoints" in r.split(os.sep):
            continue

        if "input_filenames.txt" in files:
            topics_here = []
            for filename in files:
                if filename.startswith("mallet.topic_distributions."):
                    try:
                        topics_here.append(int(filename.split(".")[-1]))
                    except ValueError:
                        pass

            run_candidates.append((r, sorted(set(topics_here))))

    if not run_candidates:
        print("No MALLET run folders found under this directory.")
        return None

    print(prompt)
    for index, (path, topics) in enumerate(run_candidates, start=1):
        relative_path = os.path.relpath(path, os.getcwd())
        suffix = (
            f" (topics: {', '.join(map(str, topics))})"
            if topics else ""
        )
        print(f"[{index}] {relative_path}{suffix}")

    while True:
        try:
            response = input(
                "Enter number, or type 'cancel' and press Enter: "
            ).strip().lower()
        except (KeyboardInterrupt, EOFError):
            print("\nSelection cancelled.")
            return None

        if response in {"cancel", "c", "esc", "escape"}:
            print("Selection cancelled.")
            return None

        try:
            choice = int(response)
        except ValueError:
            print("Invalid choice, please try again.")
            continue

        if 1 <= choice <= len(run_candidates):
            return run_candidates[choice - 1][0]

        print("Invalid choice, please try again.")

# Helper to infer available topic counts in the selected run folder, or prompt the user
# if multiple or none are found.
def infer_or_prompt_num_topics(run_dir):
    files = os.listdir(run_dir)
    candidates = []

    for filename in files:
        if filename.startswith("mallet.topic_distributions."):
            try:
                candidates.append(int(filename.split(".")[-1]))
            except ValueError:
                pass

    candidates = sorted(set(candidates))

    if not candidates:
        while True:
            try:
                response = input(
                    "Enter the number of topics, or type "
                    "'cancel' and press Enter: "
                ).strip().lower()
            except (KeyboardInterrupt, EOFError):
                print("\nSelection cancelled.")
                return None

            if response in {"cancel", "c", "esc", "escape"}:
                print("Selection cancelled.")
                return None

            try:
                return int(response)
            except ValueError:
                print("Invalid number, please try again.")

    if len(candidates) == 1:
        print(f"Detected {candidates[0]} topics in the selected run.")
        return candidates[0]

    print("Multiple topic distributions found in this run; choose one:")
    for index, topic_count in enumerate(candidates, start=1):
        print(f"[{index}] {topic_count}")

    while True:
        try:
            response = input(
                "Enter number, or type 'cancel' and press Enter: "
            ).strip().lower()
        except (KeyboardInterrupt, EOFError):
            print("\nSelection cancelled.")
            return None

        if response in {"cancel", "c", "esc", "escape"}:
            print("Selection cancelled.")
            return None

        try:
            choice = int(response)
            if 1 <= choice <= len(candidates):
                return candidates[choice - 1]
        except ValueError:
            pass

        print("Invalid choice, please try again.")
# Load the list of filenames (in modeling order) and the topic distributions
# for a given MALLET run and number of topics.
def load_model_data(run_dir, num_topics):
    """
    Load the file list (input_filenames.txt) and the doc-topic distributions
    for the specified number of topics from 'run_dir'.
    Ensures that the order of files matches how they were used in modeling.
    """
    fn_list_path = os.path.join(run_dir, "input_filenames.txt")
    if not os.path.exists(fn_list_path):
        raise FileNotFoundError(f"'input_filenames.txt' not found in {run_dir}.")

    # Read the filenames in the order used for modeling
    with open(fn_list_path, "r", encoding="utf-8") as f:
        files = [line.strip() for line in f]

    dist_file = os.path.join(run_dir, f"mallet.topic_distributions.{num_topics}")
    if not os.path.exists(dist_file):
        raise FileNotFoundError(f"No doc-topic distribution file for {num_topics} topics in {run_dir}.")

    import little_mallet_wrapper
    # Load the topic distributions for each file
    doc_topics = little_mallet_wrapper.load_topic_distributions(dist_file)
    return files, doc_topics


def _load_run_interactive():
    run_dir = choose_mallet_run(
        os.getcwd(),
        prompt="Select a MALLET run for proportions "
               "(or type 'cancel' and press Enter):",
    )
    if not run_dir:
        return None, None, None

    topic_count = infer_or_prompt_num_topics(run_dir)
    if topic_count is None:
        return None, None, None

    files, doc_topics = load_model_data(run_dir, topic_count)
    return run_dir, files, np.array(doc_topics, dtype=float)


def _try_load_token_counts(run_dir, files):
    """Attempt to load per-document token counts from sibling preprocessed data.
    Looks for preprocessed_docs.pkl in the parent folder of run_dir and verifies file order.
    Returns counts list (filtered-token counts used for modeling) or None if unavailable/mismatch.
    """
    import pickle
    parent = Path(run_dir).parent
    pkl_path = parent / "preprocessed_docs.pkl"
    if not pkl_path.exists():
        return None
    try:
        with open(pkl_path, "rb") as f:
            training_docs, token_distributions, target_files = pickle.load(f)
        # Ensure the exact file order matches the run
        if list(target_files) != list(files):
            print(
                "Warning: preprocessed_docs.pkl file list does not match run's input_filenames.txt; "
                "skipping token weighting."
            )
            return None
        # Use the FILTERED token counts that were actually passed to MALLET
        counts = [len(doc.split()) for doc in training_docs]
        return counts
    except Exception as e:
        print(f"Warning: could not read token counts from {pkl_path}: {e}")
        return None


def compute_global_topic_proportions():
    run_dir, files, doc_topics = _load_run_interactive()

    if run_dir is None:
        print("Global topic proportions cancelled.")
        return

    K = doc_topics.shape[1]

    # ...existing code...

    # Doc-weighted (simple mean across documents)
    doc_weighted = doc_topics.mean(axis=0)
    doc_weighted = doc_weighted / doc_weighted.sum()  # normalize for safety

    # Token-weighted (if filtered token counts available)
    counts = _try_load_token_counts(run_dir, files)
    token_weighted = None
    if counts is not None and len(counts) == doc_topics.shape[0]:
        weights = np.asarray(counts, dtype=float)
        total = weights.sum()
        if total > 0:
            weights = weights / total
            token_weighted = weights @ doc_topics  # shape: (K,)
            token_weighted = token_weighted / token_weighted.sum()
        else:
            print("Warning: zero total token count; skipping token-weighted proportions.")

    # Prepare Excel output
    data = {
        "topic_index": list(range(K)),
        "doc_weighted": doc_weighted.astype(float),
    }
    if token_weighted is not None:
        data["token_weighted"] = token_weighted.astype(float)

    df = pd.DataFrame(data)
    out_xlsx = os.path.join(run_dir, "global_topic_proportions.xlsx")
    df.to_excel(out_xlsx, index=False)

    # Print quick diagnostics
    print(f"Saved global proportions to: {out_xlsx}")
    print("Doc-weighted top 5 topics:")
    for t in np.argsort(-doc_weighted)[:5]:
        print(f"  Topic {t}: {doc_weighted[t]:.4f}")
    if token_weighted is not None:
        print("Token-weighted top 5 topics:")
        for t in np.argsort(-token_weighted)[:5]:
            print(f"  Topic {t}: {token_weighted[t]:.4f}")


# Run it
compute_global_topic_proportions()